# Task 2 independent specialists
Setup first; run one specialist at a time when a GPU is available. No trained classifier is needed. These are provisional baselines, not Optuna-selected specialists.


In [ ]:
import torch
print(torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# Before starting a new experiment, fetch the implementation you pushed.
# When resuming, keep the source revision used by your checkpoint.
subprocess.run(['git', 'pull', '--ff-only'], check=True)
assert (repo / 'training/train_specialists.py').is_file(), 'Push the new files first'
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)


If Colab requests a session restart after installation, restart and rerun setup. Keep the original source revision when resuming; provenance checks reject changed training sources.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
os.chdir(repo)
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)
output = Path('/content/drive/MyDrive/GenAI_A1/task2-specialists')


## Select one specialist
Start with salt. Later set condition to blur or occlusion and rerun the next cells. Each uses separate weights and output folders. Start with a five-epoch pilot; review before removing the stop limit to continue toward20. The four-hour limit applies to this one run.


In [ ]:
condition = 'salt'  # salt, blur, or occlusion
assert condition in ['salt', 'blur', 'occlusion']
specialist_output = output / condition
command = [sys.executable, '-m', 'training.train_specialists',
           '--condition', condition, '--device', 'cuda', '--max-hours', '4',
           '--stop-after-epoch', '5', '--output-dir', str(specialist_output)]
if (specialist_output / 'last.pt').exists():
    command += ['--resume', str(specialist_output / 'last.pt')]
subprocess.run(command, check=True)


In [ ]:
import json
history = json.loads((specialist_output / 'history.json').read_text())
assert history, 'No completed epoch yet; resume training'
best = min(history, key=lambda row: row['val_loss'])
print('Condition:', condition, 'Completed:', history[-1]['epoch'], 'Best epoch:', best['epoch'])
print('Validation loss:', best['val_loss'], 'L1:', best['val_l1'], 'SSIM:', best['val_ssim'])
print(best['validation']['by_condition_severity'])
from IPython.display import display, Image
display(Image(filename=str(specialist_output / 'preview.png')))


Share each specialist history.json and preview.png. Preview is the latest epoch, not necessarily the best. To continue a completed pilot, remove `--stop-after-epoch`, `5` from the command list and rerun; it resumes toward20. Preserve each entire output folder. Do not use Task1 checkpoints. Full oracle/predicted evaluation, specialist Optuna and ONNX remain pending.
